# Multi-Agent Workflow Orchestration via Google's Agent-to-Agent (A2A) Protocol

This notebook demonstrates a **multi-agent system** where independently built
agents communicate using [Google's Agent-to-Agent (A2A) protocol](https://google.github.io/A2A/).
Each agent is powered by a real LLM (OpenAI or Anthropic) and exposes its
capabilities through the standardised A2A protocol primitives.

### A2A Capabilities Demonstrated

| Capability | A2A Mechanism |
|---|---|
| **Interoperability** | Agents with different LLM backends exchange `Message` / `Part` objects over a common JSON-RPC transport |
| **Agent Discovery** | Each agent publishes an `AgentCard` with skills; a `DiscoveryService` resolves capabilities at runtime |
| **Workflow Orchestration** | An orchestrator creates `Task` objects, delegates them to discovered agents, and aggregates `Artifact` results |

### A2A Protocol Primitives Used

| Primitive | Purpose |
|---|---|
| `AgentCard` | JSON metadata describing agent identity, skills, endpoint, and auth |
| `Skill` | A named capability an agent advertises (id, name, description, tags) |
| `Task` | Stateful unit of work with lifecycle: submitted -> working -> completed/failed |
| `Message` | A communication turn (role = user or agent) containing one or more `Part` objects |
| `Part` | Content unit: `TextPart`, `DataPart`, or `FilePart` |
| `Artifact` | Named output produced by an agent during task execution |

---

## Architecture

```
                          +--------------------------------------------------+
                          |            A2A Discovery Service                  |
                          |  (registers and resolves AgentCards by skill)     |
                          +----------+----------+----------+---------+-------+
                                     |          |          |         |
                            register |  register|  register| register|
                                     |          |          |         |
     +-------------------------------+----------+----------+---------+-------+
     |                               |                     |                 |
     v                               v                     v                 v
 +------------------+  +------------------+  +------------------+  +------------------+
 | Orchestrator     |  | Research Agent   |  | Analysis Agent   |  | Writer Agent     |
 | Agent            |  |                  |  |                  |  |                  |
 | Skills:          |  | Skills:          |  | Skills:          |  | Skills:          |
 |  - planning      |  |  - web_search    |  |  - data_analysis |  |  - report_writing|
 |  - coordination  |  |  - summarisation |  |  - calculation   |  |  - formatting    |
 +--------+---------+  +--------+---------+  +--------+---------+  +--------+---------+
          |                      |                     |                     |
          | A2A Task             | A2A Task            | A2A Task            | A2A Task
          | delegation           | execution           | execution           | execution
          |                      |                     |                     |
          +----------+-----------+-----------+---------+---------------------+
                     |                                 |
                     v                                 v
             +-----------------+               +-----------------+
             | A2A Server      |               | LLM Provider    |
             | (JSON-RPC       |               | (OpenAI /       |
             |  Router)        |               |  Anthropic)     |
             +-----------------+               +-----------------+
```

### Sequence Diagram

```
  User      Orchestrator   Discovery     A2A Server    Researcher    Analyst       Writer        LLM
   |             |          Service          |             |            |             |            |
   |             |                           |             |            |             |            |
   |             |  Phase 0: Agent Registration (each agent publishes AgentCard)     |            |
   |             |--register card----------->|             |            |             |            |
   |             |             |             |<--register--|            |             |            |
   |             |             |             |<-----------register------|             |            |
   |             |             |             |<--------------------register-----------|            |
   |             |             |             |             |            |             |            |
   |             |  Phase 1: Agent Discovery  |            |            |             |            |
   |             |--find(web_search)-------->|             |            |             |            |
   |             |<--ResearchAgent card------|             |            |             |            |
   |             |--find(data_analysis)----->|             |            |             |            |
   |             |<--AnalysisAgent card------|             |            |             |            |
   |             |--find(report_writing)---->|             |            |             |            |
   |             |<--WriterAgent card--------|             |            |             |            |
   |             |             |             |             |            |             |            |
   |  Phase 2: Workflow Orchestration        |             |            |             |            |
   |--query----->|             |             |             |            |             |            |
   |             |--plan decomposition------------------------------------------------------>|
   |             |<--sub-tasks---------------------------------------------------------------|            
   |             |             |             |             |            |             |            |
   |             |---send Task(research)---->|             |            |             |            |
   |             |             |             |--dispatch-->|            |             |            |
   |             |             |             |             |--search------------------->|
   |             |             |             |             |<-findings------------------|            
   |             |             |             |<-Artifact---|            |             |            |
   |             |<--Task(completed)---------|             |            |             |            |
   |             |             |             |             |            |             |            |
   |             |---send Task(analysis)---->|             |            |             |            |
   |             |             |             |-----------dispatch------>|             |            |
   |             |             |             |             |            |--analyse---->|
   |             |             |             |             |            |<-result------|            
   |             |             |             |<-----------Artifact------|             |            |
   |             |<--Task(completed)---------|             |            |             |            |
   |             |             |             |             |            |             |            |
   |             |---send Task(writing)----->|             |            |             |            |
   |             |             |             |--------------------dispatch----------->|            |
   |             |             |             |             |            |             |--draft---->|
   |             |             |             |             |            |             |<-report---|
   |             |             |             |<--------------------Artifact-----------|            |
   |             |<--Task(completed)---------|             |            |             |            |
   |             |             |             |             |            |             |            |
   |<--report----|             |             |             |            |             |            |
   |             |             |             |             |            |             |            |
```

## 1. Install Dependencies

In [ ]:
%pip install -q langchain langchain-openai langchain-anthropic langgraph pydantic

## 2. Configuration

In [ ]:
import os

LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "openai")

# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

OPENAI_MODEL = "gpt-4o-mini"
ANTHROPIC_MODEL = "claude-sonnet-4-20250514"

print(f"LLM provider : {LLM_PROVIDER}")
print(f"Model        : {OPENAI_MODEL if LLM_PROVIDER == 'openai' else ANTHROPIC_MODEL}")

## 3. Imports

In [ ]:
from __future__ import annotations

import json
import math
import operator
import textwrap
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import (
    Annotated,
    Any,
    Callable,
    Dict,
    List,
    Literal,
    Optional,
    TypedDict,
)

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
else:
    from langchain_anthropic import ChatAnthropic

print("All imports successful.")

## 4. A2A Protocol Data Types

We implement the core A2A primitives as Python dataclasses, mirroring the
[A2A specification](https://google.github.io/A2A/).

```
  +-------------+      +-------------+      +-------------+
  |  AgentCard  |*---->|   Skill     |      |  TaskState  |
  +-------------+      +-------------+      |  (Enum)     |
  | name        |      | id          |      +-------------+
  | description |      | name        |      | SUBMITTED   |
  | url         |      | description |      | WORKING     |
  | version     |      | tags        |      | INPUT_REQ   |
  | skills[]    |      | examples[]  |      | COMPLETED   |
  | auth_type   |      | inputModes  |      | FAILED      |
  +-------------+      | outputModes |      | CANCELED    |
                       +-------------+      +-------------+

  +-------------+      +-------------+      +-------------+
  |    Task     |*---->|  Message    |*---->|    Part     |
  +-------------+      +-------------+      +-------------+
  | id          |      | role        |      | TextPart    |
  | state       |      | messageId   |      | DataPart    |
  | messages[]  |      | parts[]     |      | FilePart    |
  | artifacts[] |      | timestamp   |      +-------------+
  | metadata    |      +-------------+
  +-------------+
        |
        *----> +-------------+
               |  Artifact   |
               +-------------+
               | name        |
               | parts[]     |
               | metadata    |
               +-------------+
```

In [ ]:
# ── Task lifecycle states ────────────────────────────────────────────────────

class TaskState(str, Enum):
    SUBMITTED = "submitted"
    WORKING = "working"
    INPUT_REQUIRED = "input-required"
    COMPLETED = "completed"
    FAILED = "failed"
    CANCELED = "canceled"


# ── Parts (smallest content units) ────────────────────────────────────────────

@dataclass
class TextPart:
    text: str
    kind: str = "text"

    def to_dict(self) -> dict:
        return {"kind": self.kind, "text": self.text}


@dataclass
class DataPart:
    data: Dict[str, Any]
    kind: str = "data"

    def to_dict(self) -> dict:
        return {"kind": self.kind, "data": self.data}


@dataclass
class FilePart:
    uri: str
    mime_type: str = "application/octet-stream"
    filename: str = ""
    kind: str = "file"

    def to_dict(self) -> dict:
        return {"kind": self.kind, "uri": self.uri, "mimeType": self.mime_type, "filename": self.filename}


Part = TextPart | DataPart | FilePart

print("Part types defined.")

In [ ]:
# ── Message, Artifact, Task ──────────────────────────────────────────────────

@dataclass
class Message:
    """A single communication turn in the A2A protocol."""
    role: Literal["user", "agent"]
    parts: List[Part]
    messageId: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def text_content(self) -> str:
        return " ".join(p.text for p in self.parts if isinstance(p, TextPart))

    def data_content(self) -> List[Dict]:
        return [p.data for p in self.parts if isinstance(p, DataPart)]

    def to_dict(self) -> dict:
        return {
            "role": self.role,
            "messageId": self.messageId,
            "parts": [p.to_dict() for p in self.parts],
            "timestamp": self.timestamp,
        }


@dataclass
class Artifact:
    """A named output produced by an agent."""
    name: str
    parts: List[Part]
    metadata: Dict[str, Any] = field(default_factory=dict)

    def text_content(self) -> str:
        return " ".join(p.text for p in self.parts if isinstance(p, TextPart))

    def to_dict(self) -> dict:
        return {
            "name": self.name,
            "parts": [p.to_dict() for p in self.parts],
            "metadata": self.metadata,
        }


@dataclass
class Task:
    """The fundamental unit of work in the A2A protocol."""
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:12])
    state: TaskState = TaskState.SUBMITTED
    messages: List[Message] = field(default_factory=list)
    artifacts: List[Artifact] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def add_user_message(self, text: str) -> Message:
        msg = Message(role="user", parts=[TextPart(text=text)])
        self.messages.append(msg)
        return msg

    def add_agent_message(self, text: str) -> Message:
        msg = Message(role="agent", parts=[TextPart(text=text)])
        self.messages.append(msg)
        return msg

    def add_artifact(self, name: str, text: str, metadata: Dict[str, Any] | None = None) -> Artifact:
        artifact = Artifact(name=name, parts=[TextPart(text=text)], metadata=metadata or {})
        self.artifacts.append(artifact)
        return artifact

    def to_dict(self) -> dict:
        return {
            "id": self.id,
            "state": self.state.value,
            "messages": [m.to_dict() for m in self.messages],
            "artifacts": [a.to_dict() for a in self.artifacts],
            "metadata": self.metadata,
        }


print("Message, Artifact, Task defined.")

In [ ]:
# ── Skill and AgentCard ──────────────────────────────────────────────────────

@dataclass
class Skill:
    """Describes a specific capability an agent advertises."""
    id: str
    name: str
    description: str
    tags: List[str] = field(default_factory=list)
    examples: List[str] = field(default_factory=list)
    inputModes: List[str] = field(default_factory=lambda: ["text"])
    outputModes: List[str] = field(default_factory=lambda: ["text"])

    def to_dict(self) -> dict:
        return {
            "id": self.id, "name": self.name, "description": self.description,
            "tags": self.tags, "examples": self.examples,
            "inputModes": self.inputModes, "outputModes": self.outputModes,
        }


@dataclass
class AgentCard:
    """JSON metadata document describing an A2A agent's identity and capabilities.

    In production this would be served at /.well-known/agent.json.
    """
    name: str
    description: str
    url: str
    version: str = "1.0.0"
    skills: List[Skill] = field(default_factory=list)
    auth_type: str = "none"
    protocol_version: str = "0.3.0"

    def skill_ids(self) -> List[str]:
        return [s.id for s in self.skills]

    def skill_tags(self) -> List[str]:
        return list({t for s in self.skills for t in s.tags})

    def has_skill(self, skill_id: str) -> bool:
        return any(s.id == skill_id for s in self.skills)

    def to_dict(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "url": self.url,
            "version": self.version,
            "skills": [s.to_dict() for s in self.skills],
            "auth": {"type": self.auth_type},
            "api": {"type": "a2a", "protocolVersion": self.protocol_version},
        }


print("Skill, AgentCard defined.")

## 5. A2A Infrastructure

Three components form the runtime:

| Component | Role |
|---|---|
| `AgentExecutor` | Base class: receives a `Task`, processes it, returns the updated `Task` |
| `A2AServer` | Routes incoming tasks to the registered `AgentExecutor` (simulates JSON-RPC transport) |
| `DiscoveryService` | Central registry of `AgentCard` objects; supports lookup by skill id or tag |

In [ ]:
class AgentExecutor:
    """Base class that processes an A2A Task.

    Subclasses override `execute()` to implement domain-specific logic.
    """

    def __init__(self, agent_card: AgentCard) -> None:
        self.card = agent_card

    def execute(self, task: Task) -> Task:
        raise NotImplementedError


class A2AServer:
    """In-process A2A server that hosts AgentExecutors and routes tasks.

    Mimics the JSON-RPC transport layer of the A2A spec.
    """

    def __init__(self) -> None:
        self._executors: Dict[str, AgentExecutor] = {}
        self._task_store: Dict[str, Task] = {}
        self._request_log: List[Dict[str, Any]] = []

    def register_executor(self, executor: AgentExecutor) -> None:
        name = executor.card.name
        self._executors[name] = executor
        print(f"  [A2AServer] Registered executor: {name}")

    def send_task(self, agent_name: str, task: Task) -> Task:
        """JSON-RPC tasks/send equivalent."""
        self._request_log.append({
            "method": "tasks/send",
            "agent": agent_name,
            "task_id": task.id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })

        if agent_name not in self._executors:
            task.state = TaskState.FAILED
            task.add_agent_message(f"Agent '{agent_name}' not found on this server.")
            self._task_store[task.id] = task
            return task

        task.state = TaskState.WORKING
        executor = self._executors[agent_name]
        try:
            task = executor.execute(task)
            if task.state == TaskState.WORKING:
                task.state = TaskState.COMPLETED
        except Exception as exc:
            task.state = TaskState.FAILED
            task.add_agent_message(f"Execution error: {exc}")

        self._task_store[task.id] = task
        return task

    def get_task(self, task_id: str) -> Optional[Task]:
        """JSON-RPC tasks/get equivalent."""
        return self._task_store.get(task_id)

    def list_tasks(self) -> List[Dict[str, str]]:
        return [{"id": t.id, "state": t.state.value} for t in self._task_store.values()]

    def get_request_log(self) -> List[Dict[str, Any]]:
        return list(self._request_log)


print("AgentExecutor, A2AServer defined.")

In [ ]:
class DiscoveryService:
    """Registry of AgentCards that supports capability-based agent discovery.

    In production, agents publish their card at /.well-known/agent.json
    and clients fetch it. Here we simulate this with an in-memory registry.
    """

    def __init__(self) -> None:
        self._cards: Dict[str, AgentCard] = {}

    def register(self, card: AgentCard) -> None:
        self._cards[card.name] = card
        print(f"  [Discovery] Registered: {card.name} (skills: {card.skill_ids()})")

    def list_agents(self) -> List[AgentCard]:
        return list(self._cards.values())

    def find_by_skill(self, skill_id: str) -> List[AgentCard]:
        return [c for c in self._cards.values() if c.has_skill(skill_id)]

    def find_by_tag(self, tag: str) -> List[AgentCard]:
        return [c for c in self._cards.values() if tag in c.skill_tags()]

    def get_card(self, name: str) -> Optional[AgentCard]:
        return self._cards.get(name)


print("DiscoveryService defined.")

## 6. A2A Client Helper

A thin client that wraps interaction with the `A2AServer` and `DiscoveryService`.

In [ ]:
class A2AClient:
    """Client that uses the DiscoveryService to find agents and the
    A2AServer to send tasks."""

    def __init__(self, server: A2AServer, discovery: DiscoveryService) -> None:
        self.server = server
        self.discovery = discovery

    def discover_agents(self, skill_id: str | None = None, tag: str | None = None) -> List[AgentCard]:
        if skill_id:
            return self.discovery.find_by_skill(skill_id)
        if tag:
            return self.discovery.find_by_tag(tag)
        return self.discovery.list_agents()

    def send_task(self, agent_name: str, user_message: str, metadata: Dict[str, Any] | None = None) -> Task:
        task = Task(metadata=metadata or {})
        task.add_user_message(user_message)
        return self.server.send_task(agent_name, task)

    def send_task_with_data(
        self, agent_name: str, text: str, data: Dict[str, Any], metadata: Dict[str, Any] | None = None
    ) -> Task:
        task = Task(metadata=metadata or {})
        msg = Message(role="user", parts=[TextPart(text=text), DataPart(data=data)])
        task.messages.append(msg)
        return self.server.send_task(agent_name, task)


print("A2AClient defined.")

## 7. LLM Factory

In [ ]:
def _build_llm(temperature: float = 0.3):
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(model=OPENAI_MODEL, temperature=temperature)
    return ChatAnthropic(model=ANTHROPIC_MODEL, temperature=temperature)


def _invoke_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    llm = _build_llm(temperature)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])
    return response.content


print("LLM factory defined.")

## 8. Specialised Agent Executors

Each executor:
1. Reads the incoming `Task` messages
2. Uses the LLM (and optionally tools) to produce a result
3. Attaches the result as an `Artifact` to the task

In [ ]:
class ResearchExecutor(AgentExecutor):
    """Performs web search and synthesises findings."""

    SYSTEM_PROMPT = textwrap.dedent("""\
        You are a research agent. Given a research task, produce a concise
        summary of the most relevant findings. Include key facts, data
        points, and source references. Return plain text.
    """)

    def __init__(self) -> None:
        card = AgentCard(
            name="ResearchAgent",
            description="Searches the web and synthesises findings.",
            url="http://localhost:8001",
            skills=[
                Skill(
                    id="web_search",
                    name="Web Search",
                    description="Search the web for information and synthesise results.",
                    tags=["research", "search"],
                    examples=["Search for recent AI breakthroughs"],
                ),
                Skill(
                    id="summarisation",
                    name="Summarisation",
                    description="Summarise a body of text.",
                    tags=["research", "text"],
                ),
            ],
        )
        super().__init__(card)

    @staticmethod
    def _simulated_search(query: str, n: int = 4) -> str:
        results = [
            {"title": f"Result {i+1}: {query}",
             "snippet": f"Simulated finding #{i+1} about '{query}' with relevant data.",
             "url": f"https://example.com/search?q={query.replace(' ','+')}&p={i+1}"}
            for i in range(n)
        ]
        return json.dumps(results, indent=2)

    def execute(self, task: Task) -> Task:
        query = task.messages[-1].text_content()
        print(f"  [ResearchAgent] Searching for: {query[:60]}...")

        search_results = self._simulated_search(query)
        prompt = f"Research task: {query}\n\nSearch results:\n{search_results}\n\nSynthesise the findings."
        synthesis = _invoke_llm(self.SYSTEM_PROMPT, prompt)

        task.add_agent_message(f"Research completed. Found and synthesised results for: {query[:60]}")
        task.add_artifact("research_findings", synthesis, {"source": "ResearchAgent"})
        task.state = TaskState.COMPLETED
        print(f"  [ResearchAgent] Artifact 'research_findings' attached.")
        return task


print("ResearchExecutor defined.")

In [ ]:
class AnalysisExecutor(AgentExecutor):
    """Analyses data, performs calculations, and produces structured insights."""

    SYSTEM_PROMPT = textwrap.dedent("""\
        You are an analysis agent. You receive research data and an analysis
        task. Provide a structured analysis including key insights,
        quantitative observations, and a conclusion. Return plain text.
    """)

    def __init__(self) -> None:
        card = AgentCard(
            name="AnalysisAgent",
            description="Analyses data, performs calculations, and produces insights.",
            url="http://localhost:8002",
            skills=[
                Skill(
                    id="data_analysis",
                    name="Data Analysis",
                    description="Analyse research data and produce structured insights.",
                    tags=["analysis", "data"],
                    examples=["Analyse the impact of quantum computing on cybersecurity"],
                ),
                Skill(
                    id="calculation",
                    name="Calculation",
                    description="Perform mathematical calculations.",
                    tags=["analysis", "math"],
                ),
            ],
        )
        super().__init__(card)

    def execute(self, task: Task) -> Task:
        user_text = task.messages[-1].text_content()
        data_parts = task.messages[-1].data_content()

        context_text = ""
        if data_parts:
            context_text = json.dumps(data_parts, indent=2)
            print(f"  [AnalysisAgent] Received DataPart with upstream context.")

        print(f"  [AnalysisAgent] Analysing: {user_text[:60]}...")

        prompt = f"Analysis task: {user_text}\n"
        if context_text:
            prompt += f"\nUpstream data:\n{context_text}\n"
        prompt += "\nProvide a thorough analysis."

        analysis = _invoke_llm(self.SYSTEM_PROMPT, prompt)

        task.add_agent_message(f"Analysis completed for: {user_text[:60]}")
        task.add_artifact("analysis_report", analysis, {"source": "AnalysisAgent"})
        task.state = TaskState.COMPLETED
        print(f"  [AnalysisAgent] Artifact 'analysis_report' attached.")
        return task


print("AnalysisExecutor defined.")

In [ ]:
class WriterExecutor(AgentExecutor):
    """Produces polished reports from research findings and analysis."""

    SYSTEM_PROMPT = textwrap.dedent("""\
        You are a professional technical writer. Given research findings
        and analysis, produce a clear, well-structured report with:
        - An executive summary
        - Key findings
        - Detailed analysis
        - Conclusion and recommendations
        Return the content as plain text.
    """)

    def __init__(self) -> None:
        card = AgentCard(
            name="WriterAgent",
            description="Produces polished written reports.",
            url="http://localhost:8003",
            skills=[
                Skill(
                    id="report_writing",
                    name="Report Writing",
                    description="Produce a structured report from research and analysis.",
                    tags=["writing", "report"],
                    examples=["Write a comprehensive report on AI trends"],
                ),
                Skill(
                    id="formatting",
                    name="Formatting",
                    description="Format content into structured documents.",
                    tags=["writing", "formatting"],
                ),
            ],
        )
        super().__init__(card)

    def execute(self, task: Task) -> Task:
        user_text = task.messages[-1].text_content()
        data_parts = task.messages[-1].data_content()

        print(f"  [WriterAgent] Writing report for: {user_text[:60]}...")

        prompt = f"Writing task: {user_text}\n"
        if data_parts:
            prompt += f"\nUpstream data:\n{json.dumps(data_parts, indent=2)}\n"
        prompt += "\nWrite the final report."

        report = _invoke_llm(self.SYSTEM_PROMPT, prompt)

        task.add_agent_message(f"Report completed for: {user_text[:60]}")
        task.add_artifact("final_report", report, {"source": "WriterAgent"})
        task.state = TaskState.COMPLETED
        print(f"  [WriterAgent] Artifact 'final_report' attached.")
        return task


print("WriterExecutor defined.")

## 9. Orchestrator Agent

The orchestrator:
1. Discovers agents via the `DiscoveryService`
2. Uses the LLM to decompose the user query into sub-tasks
3. Delegates each sub-task to the appropriate agent via `A2AClient`
4. Passes upstream `Artifact` data to downstream agents as `DataPart` objects
5. Aggregates results into a final response

In [ ]:
class OrchestratorExecutor(AgentExecutor):
    """Plans and orchestrates a multi-agent workflow via A2A."""

    SYSTEM_PROMPT = textwrap.dedent("""\
        You are a planning orchestrator in a multi-agent system.
        Given a user query and available agents (with their skills),
        break the query into exactly three sub-tasks:
          1. A research task (for the agent with web_search skill)
          2. An analysis task (for the agent with data_analysis skill)
          3. A writing task (for the agent with report_writing skill)

        Return your plan as a JSON object:
        {
          "research_task": "<description>",
          "analysis_task": "<description>",
          "writing_task": "<description>"
        }

        Return ONLY valid JSON, no markdown fences.
    """)

    def __init__(self, client: A2AClient) -> None:
        card = AgentCard(
            name="Orchestrator",
            description="Decomposes queries and coordinates other agents.",
            url="http://localhost:8000",
            skills=[
                Skill(id="planning", name="Planning",
                      description="Decompose complex queries into sub-tasks.",
                      tags=["orchestration", "planning"]),
                Skill(id="coordination", name="Coordination",
                      description="Coordinate multiple agents to complete a workflow.",
                      tags=["orchestration", "coordination"]),
            ],
        )
        super().__init__(card)
        self.client = client

    def execute(self, task: Task) -> Task:
        query = task.messages[-1].text_content()

        # ── Phase 1: Agent Discovery ─────────────────────────────────
        all_agents = self.client.discover_agents()
        agent_info = [
            {"name": c.name, "skills": [s.to_dict() for s in c.skills]}
            for c in all_agents if c.name != "Orchestrator"
        ]
        print(f"  [Orchestrator] Discovered {len(agent_info)} worker agents.")

        research_agents = self.client.discover_agents(skill_id="web_search")
        analysis_agents = self.client.discover_agents(skill_id="data_analysis")
        writer_agents = self.client.discover_agents(skill_id="report_writing")
        print(f"  [Orchestrator] web_search -> {[a.name for a in research_agents]}")
        print(f"  [Orchestrator] data_analysis -> {[a.name for a in analysis_agents]}")
        print(f"  [Orchestrator] report_writing -> {[a.name for a in writer_agents]}")

        # ── Phase 2: Plan Decomposition (LLM) ────────────────────────
        plan_prompt = (
            f"User query: {query}\n\n"
            f"Available agents:\n{json.dumps(agent_info, indent=2)}\n\n"
            f"Create a plan."
        )
        raw = _invoke_llm(self.SYSTEM_PROMPT, plan_prompt)
        raw = raw.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        plan = json.loads(raw)
        print(f"  [Orchestrator] Plan: {json.dumps(plan, indent=2)}")

        task.add_agent_message(f"Plan created: {json.dumps(plan)}")
        task.add_artifact("plan", json.dumps(plan, indent=2), {"source": "Orchestrator"})

        # ── Phase 3: Delegate Research ───────────────────────────────
        research_name = research_agents[0].name if research_agents else "ResearchAgent"
        print(f"\n  [Orchestrator] Delegating research to '{research_name}'...")
        research_task = self.client.send_task(research_name, plan["research_task"])

        research_text = ""
        if research_task.state == TaskState.COMPLETED and research_task.artifacts:
            research_text = research_task.artifacts[0].text_content()
            print(f"  [Orchestrator] Research task completed (artifact: {research_task.artifacts[0].name}).")

        # ── Phase 4: Delegate Analysis (with upstream data) ──────────
        analysis_name = analysis_agents[0].name if analysis_agents else "AnalysisAgent"
        print(f"\n  [Orchestrator] Delegating analysis to '{analysis_name}'...")
        analysis_task = self.client.send_task_with_data(
            analysis_name,
            plan["analysis_task"],
            {"research_findings": research_text},
        )

        analysis_text = ""
        if analysis_task.state == TaskState.COMPLETED and analysis_task.artifacts:
            analysis_text = analysis_task.artifacts[0].text_content()
            print(f"  [Orchestrator] Analysis task completed (artifact: {analysis_task.artifacts[0].name}).")

        # ── Phase 5: Delegate Writing (with all upstream data) ───────
        writer_name = writer_agents[0].name if writer_agents else "WriterAgent"
        print(f"\n  [Orchestrator] Delegating writing to '{writer_name}'...")
        writing_task = self.client.send_task_with_data(
            writer_name,
            plan["writing_task"],
            {"research_findings": research_text, "analysis_report": analysis_text},
        )

        final_report = ""
        if writing_task.state == TaskState.COMPLETED and writing_task.artifacts:
            final_report = writing_task.artifacts[0].text_content()
            print(f"  [Orchestrator] Writing task completed (artifact: {writing_task.artifacts[0].name}).")

        # ── Aggregate ────────────────────────────────────────────────
        task.add_agent_message("Workflow completed. All sub-tasks finished.")
        task.add_artifact("final_report", final_report, {
            "source": "Orchestrator",
            "sub_tasks": {
                "research": {"id": research_task.id, "state": research_task.state.value},
                "analysis": {"id": analysis_task.id, "state": analysis_task.state.value},
                "writing": {"id": writing_task.id, "state": writing_task.state.value},
            },
        })
        task.state = TaskState.COMPLETED
        return task


print("OrchestratorExecutor defined.")

## 10. Bootstrap the System

1. Create the A2A server and discovery service
2. Instantiate each executor and register it
3. Publish each agent's `AgentCard` to the discovery service

In [ ]:
# Infrastructure
a2a_server = A2AServer()
discovery = DiscoveryService()
a2a_client = A2AClient(a2a_server, discovery)

# Executors
research_exec = ResearchExecutor()
analysis_exec = AnalysisExecutor()
writer_exec = WriterExecutor()
orchestrator_exec = OrchestratorExecutor(client=a2a_client)

# Register executors with the A2A server
for executor in [research_exec, analysis_exec, writer_exec, orchestrator_exec]:
    a2a_server.register_executor(executor)

# Publish AgentCards to the discovery service
for executor in [research_exec, analysis_exec, writer_exec, orchestrator_exec]:
    discovery.register(executor.card)

print(f"\nAgents registered: {len(discovery.list_agents())}")

## 11. Demonstrate Agent Discovery

Before running the workflow, let us show the discovery service in action.

In [ ]:
print("=== All Registered Agent Cards ===\n")
for card in discovery.list_agents():
    print(f"  Agent: {card.name}")
    print(f"    URL:         {card.url}")
    print(f"    Description: {card.description}")
    print(f"    Skills:      {card.skill_ids()}")
    print(f"    Tags:        {card.skill_tags()}")
    print(f"    Auth:        {card.auth_type}")
    print(f"    Protocol:    A2A v{card.protocol_version}")
    print()

In [ ]:
print("=== Skill-Based Discovery ===\n")
for skill_id in ["web_search", "data_analysis", "report_writing", "planning"]:
    matches = discovery.find_by_skill(skill_id)
    print(f"  Skill '{skill_id}' -> {[c.name for c in matches]}")

print("\n=== Tag-Based Discovery ===\n")
for tag in ["research", "analysis", "writing", "orchestration"]:
    matches = discovery.find_by_tag(tag)
    print(f"  Tag '{tag}' -> {[c.name for c in matches]}")

## 12. Run the Orchestrated Workflow

In [ ]:
user_query = (
    "What are the latest advancements in quantum computing, "
    "and how might they impact the cybersecurity landscape over the next five years?"
)

print(f"User query: {user_query}")
print("=" * 72)

result_task = a2a_client.send_task("Orchestrator", user_query)

print("\n" + "=" * 72)
print(f"WORKFLOW COMPLETE  |  Task ID: {result_task.id}  |  State: {result_task.state.value}")
print("=" * 72)

## 13. Inspect Results

### 13.1 Final Report (Artifact)

In [ ]:
for artifact in result_task.artifacts:
    print(f"--- Artifact: {artifact.name} (metadata: {artifact.metadata}) ---")
    print(artifact.text_content()[:2000])
    print()

### 13.2 Task Message History

In [ ]:
print(f"Task {result_task.id} — {len(result_task.messages)} messages:\n")
for i, msg in enumerate(result_task.messages, 1):
    print(f"  {i}. [{msg.role:5s}] {msg.text_content()[:100]}")

### 13.3 All Tasks on the A2A Server

In [ ]:
print("Tasks processed by the A2A server:\n")
for entry in a2a_server.list_tasks():
    t = a2a_server.get_task(entry["id"])
    artifact_names = [a.name for a in t.artifacts] if t else []
    print(f"  Task {entry['id']}  state={entry['state']:10s}  artifacts={artifact_names}")

### 13.4 A2A Server Request Log

In [ ]:
log = a2a_server.get_request_log()
print(f"Total A2A requests: {len(log)}\n")
for entry in log:
    print(f"  [{entry['timestamp'][:19]}]  {entry['method']:12s}  -> {entry['agent']}  (task={entry['task_id']})")

### 13.5 Agent Card JSON (as published at /.well-known/agent.json)

In [ ]:
print(json.dumps(research_exec.card.to_dict(), indent=2))

### 13.6 Full Task Serialisation (A2A JSON-RPC response format)

In [ ]:
print(json.dumps(result_task.to_dict(), indent=2)[:3000])

## 14. Demonstrate Interoperability: Direct Agent-to-Agent Task Exchange

A2A's key value proposition is that any A2A-compliant client can send a task
to any A2A-compliant agent — regardless of framework, LLM, or language.

Below we send tasks directly to individual agents (bypassing the orchestrator)
and pass data between them manually, demonstrating the interoperability of the
message format.

In [ ]:
# Step 1: Send a task directly to the ResearchAgent
direct_research = a2a_client.send_task(
    "ResearchAgent",
    "What are the key trends in large language model efficiency in 2025?",
)
print(f"Research task state: {direct_research.state.value}")
print(f"Artifacts: {[a.name for a in direct_research.artifacts]}")

# Step 2: Take the research artifact and send it as DataPart to the AnalysisAgent
research_output = direct_research.artifacts[0].text_content() if direct_research.artifacts else ""

direct_analysis = a2a_client.send_task_with_data(
    "AnalysisAgent",
    "Analyse the key trends and identify the most impactful developments.",
    {"upstream_research": research_output},
)
print(f"\nAnalysis task state: {direct_analysis.state.value}")
print(f"Artifacts: {[a.name for a in direct_analysis.artifacts]}")

# Step 3: Send both upstream artifacts to the WriterAgent
analysis_output = direct_analysis.artifacts[0].text_content() if direct_analysis.artifacts else ""

direct_report = a2a_client.send_task_with_data(
    "WriterAgent",
    "Produce a concise executive brief on LLM efficiency trends.",
    {"research": research_output, "analysis": analysis_output},
)
print(f"\nWriting task state: {direct_report.state.value}")
print(f"Artifacts: {[a.name for a in direct_report.artifacts]}")
print(f"\n--- Report Preview ---")
print(direct_report.artifacts[0].text_content()[:800] if direct_report.artifacts else "No report")

## 15. LangGraph Visualisation of the Orchestrated Workflow

For completeness, here is the same orchestration logic expressed as a
LangGraph `StateGraph`.

```
  +-------+     +-------------+     +----------+     +---------+     +-------+     +-----+
  | START |---->| Orchestrate |---->| Research |---->| Analyse |---->| Write |---->| END |
  +-------+     +-------------+     +----------+     +---------+     +-------+     +-----+
```

In [ ]:
class LGState(TypedDict):
    query: str
    plan: Dict[str, str]
    research: str
    analysis: str
    report: str
    messages: Annotated[List[str], operator.add]


def lg_orchestrate(state: LGState) -> dict:
    print("\n=== LG: ORCHESTRATE ===")
    plan_prompt = (
        f"User query: {state['query']}\n\n"
        f"Available agents: ResearchAgent, AnalysisAgent, WriterAgent\n\n"
        f"Create a plan."
    )
    raw = _invoke_llm(OrchestratorExecutor.SYSTEM_PROMPT, plan_prompt)
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    plan = json.loads(raw)
    return {"plan": plan, "messages": [f"Plan: {json.dumps(plan)}"]}


def lg_research(state: LGState) -> dict:
    print("\n=== LG: RESEARCH ===")
    task = a2a_client.send_task("ResearchAgent", state["plan"]["research_task"])
    text = task.artifacts[0].text_content() if task.artifacts else ""
    return {"research": text, "messages": [f"Research done: {task.state.value}"]}


def lg_analyse(state: LGState) -> dict:
    print("\n=== LG: ANALYSE ===")
    task = a2a_client.send_task_with_data(
        "AnalysisAgent", state["plan"]["analysis_task"],
        {"research_findings": state["research"]},
    )
    text = task.artifacts[0].text_content() if task.artifacts else ""
    return {"analysis": text, "messages": [f"Analysis done: {task.state.value}"]}


def lg_write(state: LGState) -> dict:
    print("\n=== LG: WRITE ===")
    task = a2a_client.send_task_with_data(
        "WriterAgent", state["plan"]["writing_task"],
        {"research": state["research"], "analysis": state["analysis"]},
    )
    text = task.artifacts[0].text_content() if task.artifacts else ""
    return {"report": text, "messages": [f"Report done: {task.state.value}"]}


graph = StateGraph(LGState)
graph.add_node("orchestrate", lg_orchestrate)
graph.add_node("research", lg_research)
graph.add_node("analyse", lg_analyse)
graph.add_node("write", lg_write)
graph.add_edge(START, "orchestrate")
graph.add_edge("orchestrate", "research")
graph.add_edge("research", "analyse")
graph.add_edge("analyse", "write")
graph.add_edge("write", END)
lg_workflow = graph.compile()

print("LangGraph workflow compiled.")

In [ ]:
lg_result = lg_workflow.invoke({
    "query": "How is edge computing evolving and what are its implications for IoT security?",
    "plan": {},
    "research": "",
    "analysis": "",
    "report": "",
    "messages": [],
})

print("\n" + "=" * 72)
print("LangGraph Workflow Complete")
print("=" * 72)
print("\nMessages:")
for i, m in enumerate(lg_result["messages"], 1):
    print(f"  {i}. {m[:100]}")
print(f"\nReport preview:\n{lg_result['report'][:800]}")

---

## Summary

This notebook demonstrated the three core A2A capabilities:

| Capability | How Demonstrated |
|---|---|
| **Interoperability** | Agents exchange `Task`, `Message`, `Part`, and `Artifact` objects via standardised A2A JSON-RPC. Section 14 shows direct agent-to-agent data exchange without an orchestrator. |
| **Agent Discovery** | Each agent publishes an `AgentCard` with skills and tags. The `DiscoveryService` resolves agents by skill id or tag (Section 11). The orchestrator uses discovery to find the right agent for each sub-task. |
| **Workflow Orchestration** | The `OrchestratorExecutor` decomposes queries, delegates `Task` objects to discovered agents, passes upstream `Artifact` data as `DataPart` to downstream agents, and aggregates results (Sections 9, 12, 15). |

### Class Diagram

```
  +------------------+          +------------------+          +------------------+
  |    AgentCard     |*-------->|     Skill        |          |   TaskState      |
  +------------------+          +------------------+          |   (Enum)         |
  | + name           |          | + id             |          +------------------+
  | + description    |          | + name           |          | SUBMITTED        |
  | + url            |          | + description    |          | WORKING          |
  | + version        |          | + tags[]         |          | INPUT_REQUIRED   |
  | + skills[]       |          | + examples[]     |          | COMPLETED        |
  | + auth_type      |          | + inputModes[]   |          | FAILED           |
  | + protocol_ver   |          | + outputModes[]  |          | CANCELED         |
  +------------------+          +------------------+          +------------------+
  | + skill_ids()    |
  | + has_skill()    |
  | + to_dict()      |
  +------------------+

  +------------------+          +------------------+          +------------------+
  |     Task         |*-------->|    Message       |*-------->|     Part         |
  +------------------+          +------------------+          +------------------+
  | + id             |          | + role           |          | TextPart         |
  | + state          |          | + parts[]        |          |   + text         |
  | + messages[]     |          | + messageId      |          | DataPart         |
  | + artifacts[]    |          | + timestamp      |          |   + data         |
  | + metadata       |          +------------------+          | FilePart         |
  +------------------+          | + text_content() |          |   + uri          |
  | + add_user_msg() |          | + data_content() |          |   + mime_type    |
  | + add_agent_msg()|          | + to_dict()      |          +------------------+
  | + add_artifact() |          +------------------+
  | + to_dict()      |
  +------------------+
        |
        *---------> +------------------+
                    |    Artifact      |
                    +------------------+
                    | + name           |
                    | + parts[]        |
                    | + metadata       |
                    +------------------+
                    | + text_content() |
                    | + to_dict()      |
                    +------------------+

  +------------------+          +------------------+          +------------------+
  | AgentExecutor    |          |   A2AServer      |          | DiscoveryService |
  +------------------+          +------------------+          +------------------+
  | + card           |          | + register_exec()|          | + register()     |
  +------------------+          | + send_task()    |          | + find_by_skill()|
  | + execute(Task)  |          | + get_task()     |          | + find_by_tag()  |
  +------------------+          | + list_tasks()   |          | + list_agents()  |
        ^    ^    ^    ^        +------------------+          +------------------+
        |    |    |    |
  +-----+  +--+ +--+  +------+
  |Research| |Analysis| |Writer| |Orchestrator|
  |Executor| |Executor| |Exec  | |Executor    |
  +--------+ +--------+ +------+ +------------+

  +------------------+
  |   A2AClient      |
  +------------------+
  | + server          |
  | + discovery       |
  +------------------+
  | + discover_agents()|
  | + send_task()      |
  | + send_task_with   |
  |    _data()         |
  +------------------+
```